# Modelo de Previsão do Índice de Ilha de Calor Urbana (UHI)

Este notebook demonstra o processo de desenvolvimento de um modelo para prever o Índice de Ilha de Calor Urbana (UHI) utilizando dados de satélite como features, sem usar coordenadas geográficas como preditores.


## 1. Configuração do Ambiente e Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
warnings.filterwarnings('ignore')

# Configurações para visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 2. Carregamento e Exploração dos Dados

Vamos carregar o conjunto de dados de treinamento e explorar sua estrutura:

In [ ]:
# Carregar os dados de treinamento
df_train = pd.read_csv('dataset/training_data.csv')

# Exibir as primeiras linhas
print("Primeiras linhas do conjunto de dados:")
df_train.head()

In [ ]:
# Informações sobre as colunas
print("\nInformações sobre o conjunto de dados:")
df_train.info()

In [ ]:
# Estatísticas descritivas
print("\nEstatísticas descritivas:")
df_train.describe()

## 3. Pré-processamento dos Dados

In [ ]:
# Remover a coluna de timestamp
df_train = df_train.drop('datetime', axis=1)

# Verificar as colunas restantes
print("Colunas após remoção do timestamp:")
df_train.columns

In [ ]:
# Verificar valores nulos
print("\nValores nulos por coluna:")
df_train.isnull().sum()

## 4. Análise Exploratória dos Dados (EDA)

Vamos visualizar a distribuição do Índice UHI e explorar sua relação espacial:

In [ ]:
# Histograma do Índice UHI
plt.figure(figsize=(10, 6))
sns.histplot(df_train['UHI Index'], kde=True)
plt.title('Distribuição do Índice de Ilha de Calor Urbana (UHI)')
plt.xlabel('Índice UHI')
plt.ylabel('Frequência')
plt.grid(True)
plt.show()

In [ ]:
# Scatterplot das localizações coloridas pelo Índice UHI
plt.figure(figsize=(12, 10))
scatter = plt.scatter(df_train['Longitude'], df_train['Latitude'], 
                    c=df_train['UHI Index'], cmap='coolwarm', 
                    alpha=0.6, s=10)
plt.colorbar(scatter, label='Índice UHI')
plt.title('Distribuição Espacial do Índice UHI')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()

In [ ]:
# Estatísticas por quartis do Índice UHI
print("Estatísticas por quartis do Índice UHI:")
df_train['UHI_quartil'] = pd.qcut(df_train['UHI Index'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_train.groupby('UHI_quartil')['UHI Index'].describe()

## 5. Extração de Dados de Satélite

Agora vamos configurar o ambiente para extrair dados de satélite do Microsoft Planetary Computer. Precisaremos dos dados do Sentinel-2 e Landsat que correspondam temporalmente à coleta dos dados UHI (24 de julho de 2021).


In [ ]:
import planetary_computer
from pystac_client import Client
import stackstac
import rasterio
import xarray as xr
from odc.stac import stac_load
from datetime import datetime, timedelta
import pystac_client

# Verificar a versão do stackstac para debugging
import importlib.metadata
print(f"Versão do stackstac: {importlib.metadata.version('stackstac')}")

In [ ]:
# Inicializar o cliente Planetary Computer
pc = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)

# Definir intervalo de tempo para busca de imagens
# Podemos buscar imagens alguns dias antes da coleta para garantir cobertura de nuvens mínima
data_coleta = "2021-07-24"
data_inicio = (datetime.strptime(data_coleta, "%Y-%m-%d") - timedelta(days=30)).strftime("%Y-%m-%d")
data_fim = (datetime.strptime(data_coleta, "%Y-%m-%d") + timedelta(days=5)).strftime("%Y-%m-%d")

# Extrair coordenadas para definir a área de busca
min_lon = df_train['Longitude'].min()
max_lon = df_train['Longitude'].max()
min_lat = df_train['Latitude'].min()
max_lat = df_train['Latitude'].max()

# Definir o bounding box (bbox) para a área de estudo
bbox = [min_lon, min_lat, max_lon, max_lat]

print(f"Período de busca: {data_inicio} a {data_fim}")
print(f"Área de estudo (bbox): {bbox}")

### 5.2 Extração de Dados do Sentinel-2

In [ ]:
# Buscar coleção Sentinel-2 Level 2A (reflectância de superfície)
search_sentinel = pc.search(
    collections=["sentinel-2-l2a"],
    datetime=f"{data_inicio}/{data_fim}",
    bbox=bbox,
    query={"eo:cloud_cover": {"lt": 20}}  # Filtrar imagens com menos de 20% de cobertura de nuvens
)

# Verificar número de itens encontrados
sentinel_items = list(search_sentinel.get_items())
print(f"Número de cenas Sentinel-2 encontradas: {len(sentinel_items)}")

# Visualizar metadados e assets disponíveis da primeira cena
if len(sentinel_items) > 0:
    print("Metadados da primeira cena Sentinel-2:")
    print(f"ID: {sentinel_items[0].id}")
    print(f"Data: {sentinel_items[0].properties['datetime']}")
    print(f"Cobertura de nuvens: {sentinel_items[0].properties['eo:cloud_cover']}%")
    
    # Verificar os assets disponíveis
    print("\nAssets disponíveis na primeira cena:")
    for asset_key in sentinel_items[0].assets.keys():
        print(f"- {asset_key}")

In [ ]:
signed_items = [planetary_computer.sign(item).to_dict() for item in sentinel_items]

In [ ]:
# Define the pixel resolution for the final product
# Define the scale according to our selected crs, so we will use degrees
resolution = 10  # meters per pixel 
scale = resolution / 111320.0 # degrees per pixel for crs=4326 

In [ ]:
data = stac_load(
    sentinel_items,
    bands=["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"],
    crs="EPSG:4326", # Latitude-Longitude
    resolution=scale, # Degrees
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
# View the dimensions of our XARRAY and the loaded variables
# This insures we have the right coordinates and spectral bands in our xarray
display(data)

In [ ]:
# Plot sample images from the time series
plot_data = data[["B04","B03","B02"]].to_array()
plot_data.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=2500)
plt.show()

In [ ]:
# Plot an RGB image for a single date
fig, ax = plt.subplots(figsize=(6,6))
plot_data.isel(time=1).plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.set_title("RGB Single Date: July 24, 2021")
ax.axis('off')
plt.show()

In [ ]:
median = data.median(dim="time").compute()

In [ ]:
# Plot an RGB image for the median composite or mosaic
# Notice how this new image is void of clouds due to statistical filtering
fig, ax = plt.subplots(figsize=(6,6))
median[["B04", "B03", "B02"]].to_array().plot.imshow(robust=True, ax=ax, vmin=0, vmax=2500)
ax.set_title("RGB Median Composite")
ax.axis('off')
plt.show()

In [ ]:
# Calculate NDVI for the median mosaic
ndvi_median = (median.B08-median.B04)/(median.B08+median.B04)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ndvi_median.plot.imshow(vmin=0.0, vmax=1.0, cmap="RdYlGn")
plt.title("Median NDVI")
plt.axis('off')
plt.show()

In [ ]:
# Calculate NDBI for the median mosaic
ndbi_median = (median.B11-median.B08)/(median.B11+median.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ndbi_median.plot.imshow(vmin=-0.1, vmax=0.1, cmap="jet")
plt.title("Median NDBI")
plt.axis('off')
plt.show()

In [ ]:
# Calculate NDWI for the median mosaic
ndwi_median = (median.B03-median.B08)/(median.B03+median.B08)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ndwi_median.plot.imshow(vmin=-0.3, vmax=0.3, cmap="RdBu")
plt.title("Median NDWI")
plt.axis('off')
plt.show()

### 5.3 Extração de Dados do Landsat

In [ ]:
time_window = "2021-06-01/2021-09-01"

stac = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace
)
landsat_items = stac.search(
    bbox=bbox, 
    datetime=time_window,
    collections=["landsat-c2-l2"],
    query={"eo:cloud_cover": {"lt": 50},"platform": {"in": ["landsat-8"]}},
)

# Verificar número de itens encontrados
landsat_items = list(landsat_items.get_items())
print(f"Número de cenas Sentinel-2 encontradas: {len(landsat_items)}")

# Visualizar metadados e assets disponíveis da primeira cena
if len(landsat_items) > 0:
    print("Metadados da primeira cena Sentinel-2:")
    print(f"ID: {landsat_items[0].id}")
    print(f"Data: {landsat_items[0].properties['datetime']}")
    print(f"Cobertura de nuvens: {landsat_items[0].properties['eo:cloud_cover']}%")
    
    # Verificar os assets disponíveis
    print("\nAssets disponíveis na primeira cena:")
    for asset_key in landsat_items[0].assets.keys():
        print(f"- {asset_key}")

In [ ]:
signed_items = [planetary_computer.sign(item).to_dict() for item in landsat_items]

In [ ]:
data1 = stac_load(
    landsat_items,
    bands=["red", "green", "blue", "nir08"],
    crs="EPSG:4326", # Latitude-Longitude
    resolution=scale, # Degrees
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
data2 = stac_load(
    landsat_items,
    bands=["lwir11"],
    crs="EPSG:4326", # Latitude-Longitude
    resolution=scale, # Degrees
    chunks={"x": 2048, "y": 2048},
    dtype="uint16",
    patch_url=planetary_computer.sign,
    bbox=bbox
)

In [ ]:
# View the dimensions of our XARRAY and the loaded variables
# This insures we have the right coordinates and spectral bands in our xarray
display(data1)
display(data2)

In [ ]:
# Persist the data in memory for faster operations
data1 = data1.persist()
data2 = data2.persist()

In [ ]:
# Scale Factors for the RGB and NIR bands 
scale1 = 0.0000275 
offset1 = -0.2 
data1 = data1.astype(float) * scale1 + offset1

In [ ]:
# Scale Factors for the Surface Temperature band
scale2 = 0.00341802 
offset2 = 149.0 
kelvin_celsius = 273.15 # convert from Kelvin to Celsius
data2 = data2.astype(float) * scale2 + offset2 - kelvin_celsius

In [ ]:
plot_data = data1[["red","green","blue"]].to_array()
plot_data.plot.imshow(col='time', col_wrap=4, robust=True, vmin=0, vmax=0.25)
plt.show()

In [ ]:
# Pick one of the scenes above (numbering starts with 0)
scene = 2

In [ ]:
# Plot an RGB Real Color Image for a single date
fig, ax = plt.subplots(figsize=(9,10))
data1.isel(time=scene)[["red", "green", "blue"]].to_array().plot.imshow(robust=True, ax=ax, vmin=0.0, vmax=0.25)
ax.set_title("RGB Real Color")
ax.axis('off')
plt.show()

In [ ]:
# Calculate NDVI for the median mosaic
ndvi_data = (data1.isel(time=scene).nir08-data1.isel(time=scene).red)/(data1.isel(time=scene).nir08+data1.isel(time=scene).red)

In [ ]:
fig, ax = plt.subplots(figsize=(11,10))
ndvi_data.plot.imshow(vmin=0.0, vmax=1.0, cmap="RdYlGn")
plt.title("Vegetation Index = NDVI")
plt.axis('off')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11,10))
data2.isel(time=scene).lwir11.plot.imshow(vmin=20.0, vmax=45.0, cmap="jet")
plt.title("Land Surface Temperature (LST)")
plt.axis('off')
plt.show()